In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# التحويل باستخدام مديري التمريرات

<details>
<summary><b>إصدارات الحزم</b></summary>

الكود في هذه الصفحة طُوِّر باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

الطريقة الموصى بها لتحويل دائرة ما هي إنشاء مدير تمريرات متدرّج ثم تنفيذ طريقة `run` الخاصة به مع الدائرة كمدخل. تشرح هذه الصفحة كيفية تحويل دوائر الكم بهذه الطريقة.

## ما هو مدير التمريرات (المتدرّج)؟
في سياق Qiskit SDK، يشير التحويل (Transpilation) إلى عملية تحويل دائرة مدخلة إلى شكل مناسب للتنفيذ على جهاز كمي. يحدث التحويل عادةً في سلسلة من الخطوات تُسمّى تمريرات المُحوِّل (transpiler passes). تُعالَج الدائرة بواسطة كل تمريرة بالتسلسل، إذ يصبح مخرج تمريرة ما مدخلًا للتمريرة التالية. على سبيل المثال، قد تمر إحدى التمريرات عبر الدائرة وتدمج كل التسلسلات المتتالية من بوابات Qubit الفردية، ثم تقوم التمريرة التالية بتوليف هذه البوابات إلى مجموعة الأساس للجهاز المستهدف. تمريرات المُحوِّل المضمّنة في Qiskit موجودة في وحدة [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

مدير التمريرات هو كائن يخزّن قائمة من تمريرات المُحوِّل ويمكنه تنفيذها على دائرة. أنشئ مدير تمريرات عن طريق تهيئة [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) بقائمة من تمريرات المُحوِّل. لتشغيل التحويل على دائرة، استدعِ طريقة [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) مع الدائرة كمدخل.

مدير التمريرات المتدرّج هو نوع خاص من مديري التمريرات يمثّل مستوى تجريد أعلى من مدير التمريرات العادي. بينما يتكوّن مدير التمريرات العادي من عدة تمريرات مُحوِّل، يتكوّن مدير التمريرات المتدرّج من عدة *مديري تمريرات*. هذا تجريد مفيد لأن التحويل يحدث عادةً في مراحل منفصلة، كما هو موضّح في [مراحل المُحوِّل](transpiler-stages)، حيث تُمثَّل كل مرحلة بمدير تمريرات. يُمثَّل مديرو التمريرات المتدرّجة بالفئة [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager). يصف باقي هذه الصفحة كيفية إنشاء وتخصيص مديري التمريرات (المتدرّجة).

## توليد مدير تمريرات متدرّج مُعدَّ مسبقًا
لإنشاء مدير تمريرات متدرّج مُعدَّ مسبقًا بإعدادات افتراضية معقولة، استخدم دالة [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager):

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

لتحويل دائرة أو قائمة من الدوائر باستخدام مدير التمريرات، مرّر الدائرة أو قائمة الدوائر إلى طريقة `run`. لنجرّب ذلك على دائرة ثنائية Qubit تتكوّن من بوابة Hadamard يتبعها بوابتا CX متجاورتان:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/c1426e6c-f506-4938-8c0a-05198bae9746-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

اطّلع على [الإعدادات الافتراضية للتحويل وخيارات الضبط](defaults-and-configuration-options) للاطلاع على وصف المعاملات الممكنة لدالة `generate_preset_pass_manager`. معاملات `generate_preset_pass_manager` تطابق معاملات دالة [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="هل تجد صعوبة في تذكّر تفاصيل مدير التمريرات؟ جرّب أن تسأل Qiskit Code Assistant." />

إذا لم تُلبِّ مديرو التمريرات المُعدَّة مسبقًا احتياجاتك، فيمكنك تخصيص التحويل عن طريق إنشاء مديري تمريرات (متدرّجة) أو حتى تمريرات تحويل. يصف باقي هذه الصفحة كيفية إنشاء مديري التمريرات. للاطلاع على تعليمات إنشاء تمريرات التحويل، راجع [اكتب تمريرة مُحوِّل خاصة بك](custom-transpiler-pass).

## إنشاء مدير تمريراتك الخاص

تحتوي وحدة [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) على العديد من تمريرات المُحوِّل التي يمكن استخدامها لإنشاء مديري التمريرات. لإنشاء مدير تمريرات، هيّئ `PassManager` بقائمة من التمريرات. على سبيل المثال، ينشئ الكود التالي تمريرة مُحوِّل تدمج بوابات Qubit الثنائية المتجاورة ثم تولّفها إلى أساس من بوابات [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate) و[$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate) و[$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate).

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

لإثبات عمل مدير التمريرات هذا، اختبره على دائرة ثنائية Qubit تتكوّن من بوابة Hadamard يتبعها بوابتا CX متجاورتان:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/01317c54-68b5-4e41-893f-82ee223e22f0-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

لتشغيل مدير التمريرات على الدائرة، استدعِ طريقة `run`.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/019ad99b-bd38-4217-90ee-da43959dc8ad-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

للاطلاع على مثال أكثر تقدّمًا يوضّح كيفية إنشاء مدير تمريرات لتطبيق تقنية قمع الأخطاء المعروفة بالفصل الديناميكي، راجع [إنشاء مدير تمريرات للفصل الديناميكي](dynamical-decoupling-pass-manager).

## إنشاء مدير تمريرات متدرّج

`StagedPassManager` هو مدير تمريرات مؤلَّف من مراحل فردية، حيث تُعرَّف كل مرحلة بنسخة `PassManager`. يمكنك إنشاء `StagedPassManager` عن طريق تحديد المراحل المطلوبة. على سبيل المثال، ينشئ الكود التالي مدير تمريرات متدرّجًا بمرحلتين: `init` و`translation`. تُعرَّف مرحلة `translation` بمدير التمريرات الذي أُنشئ سابقًا.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

لا يوجد حدٌّ لعدد المراحل التي يمكنك إضافتها في مدير التمريرات المتدرّج.

طريقة أخرى مفيدة لإنشاء مدير تمريرات متدرّج هي البدء بمدير تمريرات متدرّج مُعدَّ مسبقًا ثم استبدال بعض المراحل. على سبيل المثال، ينشئ الكود التالي مدير تمريرات مُعدَّ مسبقًا بمستوى تحسين 3، ثم يحدد مرحلة `pre_layout` مخصصة.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

قد تكون [دوال مولّد المراحل](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) مفيدة لبناء مديري التمريرات المخصصة.
تولّد هذه الدوال مراحل توفر وظائف شائعة تُستخدم في كثير من مديري التمريرات.
على سبيل المثال، يمكن استخدام [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) لتوليد مرحلة
"تضمين" `Layout` أولي مختار من تمريرة تخطيط إلى الجهاز المستهدف المحدد.

## الخطوات التالية
> **Tip:** - [اكتب تمريرة مُحوِّل مخصصة](custom-transpiler-pass).
>     - [إنشاء مدير تمريرات للفصل الديناميكي](dynamical-decoupling-pass-manager).
>     - لمعرفة المزيد عن دالة `generate_preset_passmanager`، راجع موضوع [الإعدادات الافتراضية للتحويل وخيارات الضبط](defaults-and-configuration-options).
>     - جرّب دليل [مقارنة إعدادات المُحوِّل](/guides/circuit-transpilation-settings).
>     - راجع [توثيق واجهة برمجة المُحوِّل.](https://docs.quantum.ibm.com/api/qiskit/transpiler)